
# 03 - Pose Landmark Extraction

## Objective
Extract human body pose landmarks from workout videos using MediaPipe Pose Landmarker.

## Dataset
- 652 workout videos
- 22 exercise categories

## Initial Plan
1. Load the dataset and pretrained pose model.
2. Sample frames from each video.
3. Detect body landmarks in each sampled frame.
4. Save landmark coordinates and detection information to CSV files.
5. Evaluate detection quality before using the extracted data for movement analysis.

## Important Limitation
Pose landmarks describe body position. They do not directly establish whether an exercise causes an injury or whether a person will get injured.


In [3]:

from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd

from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import json

In [4]:

project_folder = Path("..")

dataset_folder = project_folder / "data" / "archive(5)"
model_path = project_folder / "models" / "pose_landmarker_lite.task"
output_folder = project_folder / "outputs"

output_folder.mkdir(parents=True, exist_ok=True)

print("Dataset folder:", dataset_folder.resolve())
print("Model exists:", model_path.exists())
print("Dataset exists:", dataset_folder.exists())
print("Output folder:", output_folder.resolve())


Dataset folder: C:\athlete-performance-injury-risk\data\archive(5)
Model exists: True
Dataset exists: True
Output folder: C:\athlete-performance-injury-risk\outputs


In [5]:
import sys
print(sys.executable)


c:\athlete-performance-injury-risk\.venv\Scripts\python.exe


In [6]:
video_extensions = {".mp4", ".mov", ".avi", ".mkv"}
video_files = sorted( path
                     for path in dataset_folder.rglob("*")
                     if path.is_file() and path.suffix.lower() in video_extensions )
print("Total videos found ", len(video_files))

Total videos found  652


In [7]:
video_records = []
for video_path in video_files:
    category = video_path.parent.name
    video_records.append({
        "video_path": str(video_path),
        "video_name": video_path.name,
        "category": category
    })
video_df = pd.DataFrame(video_records)
print("Total videos:", len(video_df))
print("Total categories:", video_df["category"].nunique())

video_df.head()

Total videos: 652
Total categories: 22


,video_path,video_name,category
0,..\data\archive(5)\barbell biceps curl\barbell...,barbell biceps curl_1.mp4,barbell biceps curl
1,..\data\archive(5)\barbell biceps curl\barbell...,barbell biceps curl_10.mp4,barbell biceps curl
2,..\data\archive(5)\barbell biceps curl\barbell...,barbell biceps curl_11.mp4,barbell biceps curl
3,..\data\archive(5)\barbell biceps curl\barbell...,barbell biceps curl_12.mp4,barbell biceps curl
4,..\data\archive(5)\barbell biceps curl\barbell...,barbell biceps curl_13.mp4,barbell biceps curl


In [8]:
category_counts = video_df["category"].value_counts().sort_values(ascending=False)
print(category_counts)

category
barbell biceps curl    62
bench press            61
push-up                56
lat pulldown           51
tricep Pushdown        50
lateral raise          37
incline bench press    33
deadlift               32
squat                  29
chest fly machine      28
pull Up                26
leg extension          25
leg raises             21
t bar row              21
tricep dips            20
hammer curl            19
hip thrust             18
shoulder press         17
romanian deadlift      14
russian twist          13
decline bench press    12
plank                   7
Name: count, dtype: int64


In [9]:
FRAMES_PER_VIDEO = 5

checkpoint_folder = output_folder / "landmark_checkpoints"
checkpoint_folder.mkdir(parents=True, exist_ok=True)

In [10]:
if "pose_landmarker" not in globals():
    base_options = python.BaseOptions(
        model_asset_path=str(model_path)
    )
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.IMAGE,
        num_poses=1
    )
    pose_landmarker = vision.PoseLandmarker.create_from_options(
        options
    )

In [12]:
all_landmarks = []
all_frames = []
failed_videos = []

total_videos = len(video_df)

for video_index, row in video_df.reset_index(drop=True).iterrows():

    video_path = Path(row["video_path"])
    checkpoint_path = checkpoint_folder / f"video_{video_index:04d}.json"

    # Skip videos already completed
    if checkpoint_path.exists():
        with open(checkpoint_path, "r", encoding="utf-8") as file:
            saved_data = json.load(file)

        all_landmarks.extend(saved_data["landmarks"])
        all_frames.extend(saved_data["frames"])

        print(f"[{video_index + 1}/{total_videos}] Skipped: {video_path.name}")
        continue

    video_landmarks = []
    video_frames = []

    cap = None

    try:
        cap = cv2.VideoCapture(str(video_path))

        if not cap.isOpened():
            raise RuntimeError("Could not open video")

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames <= 0:
            raise RuntimeError("Video has no readable frames")

        frame_positions = np.linspace(
            0,
            total_frames - 1,
            num=min(FRAMES_PER_VIDEO, total_frames),
            dtype=int
        )

        for frame_number in frame_positions:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_number))
            success, frame = cap.read()

            if not success:
                video_frames.append({
                    "video_name": video_path.name,
                    "category": row["category"],
                    "frame_number": int(frame_number),
                    "pose_detected": False,
                    "status": "frame_read_failed"
                })
                continue

            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=frame_rgb
            )

            result = pose_landmarker.detect(mp_image)
            pose_found = len(result.pose_landmarks) > 0

            video_frames.append({
                "video_name": video_path.name,
                "category": row["category"],
                "frame_number": int(frame_number),
                "pose_detected": pose_found,
                "status": "ok" if pose_found else "no_pose_detected"
            })

            if pose_found:
                for landmark_id, landmark in enumerate(
                    result.pose_landmarks[0]
                ):
                    video_landmarks.append({
                        "video_name": video_path.name,
                        "category": row["category"],
                        "frame_number": int(frame_number),
                        "landmark_id": landmark_id,
                        "x": landmark.x,
                        "y": landmark.y,
                        "z": landmark.z,
                        "visibility": landmark.visibility
                    })

        # Save this video's checkpoint only after it finishes
        saved_data = {
            "landmarks": video_landmarks,
            "frames": video_frames
        }

        with open(checkpoint_path, "w", encoding="utf-8") as file:
            json.dump(saved_data, file)

        all_landmarks.extend(video_landmarks)
        all_frames.extend(video_frames)

        print(
            f"[{video_index + 1}/{total_videos}] "
            f"Completed: {video_path.name} | "
            f"Frames: {len(video_frames)} | "
            f"Poses: {sum(f['pose_detected'] for f in video_frames)}"
        )

    except Exception as error:
        failed_videos.append({
            "video_name": video_path.name,
            "video_path": str(video_path),
            "error": str(error)
        })
        print(f"[{video_index + 1}/{total_videos}] FAILED: {video_path.name} | {error}")

    finally:
        if cap is not None:
            cap.release()


[1/652] Skipped: barbell biceps curl_1.mp4
[2/652] Skipped: barbell biceps curl_10.mp4
[3/652] Skipped: barbell biceps curl_11.mp4
[4/652] Skipped: barbell biceps curl_12.mp4
[5/652] Skipped: barbell biceps curl_13.mp4
[6/652] Skipped: barbell biceps curl_14.mp4
[7/652] Skipped: barbell biceps curl_15.mp4
[8/652] Skipped: barbell biceps curl_16.mp4
[9/652] Skipped: barbell biceps curl_17.mp4
[10/652] Skipped: barbell biceps curl_18.mp4
[11/652] Skipped: barbell biceps curl_19.mp4
[12/652] Skipped: barbell biceps curl_2.mp4
[13/652] Skipped: barbell biceps curl_20.mp4
[14/652] Skipped: barbell biceps curl_21.mp4
[15/652] Skipped: barbell biceps curl_22.mp4
[16/652] Skipped: barbell biceps curl_23.mp4
[17/652] Skipped: barbell biceps curl_24.mp4
[18/652] Skipped: barbell biceps curl_25.mp4
[19/652] Skipped: barbell biceps curl_26.mp4
[20/652] Skipped: barbell biceps curl_27.mp4
[21/652] Skipped: barbell biceps curl_28.mp4
[22/652] Skipped: barbell biceps curl_29.mp4
[23/652] Skipped: bar

In [13]:
# Save combined outputs
landmarks_df = pd.DataFrame(all_landmarks)
frames_df = pd.DataFrame(all_frames)
failed_df = pd.DataFrame(failed_videos)

landmarks_df.to_csv(
    output_folder / "landmarks_all.csv",
    index=False
)
frames_df.to_csv(
    output_folder / "pose_frame_summary_all.csv",
    index=False
)
failed_df.to_csv(
    output_folder / "failed_videos.csv",
    index=False
)

In [14]:
print("\nEXTRACTION SUMMARY")
print("Videos in dataset:", total_videos)
print("Videos with recorded failures this run:", len(failed_df))
print("Frames processed:", len(frames_df))
print("Frames with pose:", int(frames_df["pose_detected"].sum()) if not frames_df.empty else 0)
print("Landmark rows:", len(landmarks_df))
print("Landmarks file:", output_folder / "landmarks_all.csv")
print("Frame summary:", output_folder / "pose_frame_summary_all.csv")
print("Failure report:", output_folder / "failed_videos.csv")



EXTRACTION SUMMARY
Videos in dataset: 652
Videos with recorded failures this run: 0
Frames processed: 3260
Frames with pose: 3005
Landmark rows: 99165
Landmarks file: ..\outputs\landmarks_all.csv
Frame summary: ..\outputs\pose_frame_summary_all.csv
Failure report: ..\outputs\failed_videos.csv


In [15]:

landmarks_path = output_folder / "landmarks_all.csv"
frames_path = output_folder / "pose_frame_summary_all.csv"
failed_path = output_folder / "failed_videos.csv"

print("Landmarks CSV exists:", landmarks_path.exists())
print("Frame summary exists:", frames_path.exists())
print("Failure report exists:", failed_path.exists())

if frames_path.exists():
    frames_df = pd.read_csv(frames_path)

    print("Total sampled frames:", len(frames_df))
    print("Frames with pose:", int(frames_df["pose_detected"].sum()))
    print("Unique videos processed:", frames_df["video_name"].nunique())

if landmarks_path.exists():
    landmarks_df = pd.read_csv(landmarks_path)

    print("Total landmark rows:", len(landmarks_df))
    print("Unique landmarks:", landmarks_df["landmark_id"].nunique())


Landmarks CSV exists: True
Frame summary exists: True
Failure report exists: True
Total sampled frames: 3260
Frames with pose: 3005
Unique videos processed: 652
Total landmark rows: 99165
Unique landmarks: 33
